In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. データの読み込みと確認

初めに使用するライブラリを読み込みます。<br>
[numpy](https://numpy.org/doc/1.21/index.html#)
[pandas](https://pandas.pydata.org/docs/#)
[matplotlib](https://matplotlib.org/stable/index.html)
[seaborn](https://seaborn.pydata.org/)
[sklearn](https://scikit-learn.org/stable/index.html)<br>
[sklearn.ensemble](https://scikit-learn.org/stable/modules/classes.html?highlight=ensemble#module-sklearn.ensemble)
[sklearn.linear_model](https://scikit-learn.org/stable/modules/classes.html?highlight=linear_model#module-sklearn.linear_model)
[sklearn.svm](https://scikit-learn.org/stable/modules/classes.html?highlight=svm#module-sklearn.svm)<br>
[sklearn.model_selection](https://scikit-learn.org/stable/modules/classes.html?highlight=model_selection#module-sklearn.model_selection)
[sklearn.preprocessing](https://scikit-learn.org/stable/modules/classes.html?highlight=preprocessing#module-sklearn.preprocessing)

In [ ]:
# ライブラリの読み込み
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

# モデル作成のためのライブラリ
from sklearn.ensemble import RandomForestClassifier

# モデル作成に役立つライブラリ
from sklearn.model_selection import train_test_split

# 性能指標(正解率)
from sklearn.metrics import accuracy_score

# 不要な警告を無視する
import warnings
warnings.filterwarnings('ignore')

pandasのread_csv関数を用いて、分析する訓練データtrain.csvとテストデータtest.csvを読み込みます。<br>
[pd.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html?highlight=read_csv#pandas.read_csv)

In [ ]:
# 学習用データ、テストデータ、提出サンプルデータの読み込み
train = pd.read_csv('../input/titanic/train.csv')
test = pd.read_csv('../input/titanic/test.csv')
sample = pd.read_csv('../input/titanic/gender_submission.csv')

データを見ていく上で、まず初めにデータのサイズを確認します。<br>
[pd.DataFrame.shape](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html?highlight=shape#pandas.DataFrame.shape)

In [ ]:
# データサイズの確認
print('学習データのサイズ:', train.shape)
print('テストデータのサイズ:', test.shape)

データの情報を確認します。<br>
[pd.DataFrame.info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html?highlight=info#pandas.DataFrame.info)

In [ ]:
# データの情報の確認
print(train.info(), '\n')
print(test.info())

学習データ、テストデータ、提出サンプルデータについて先頭の5行を表示します。<br>
[pd.DataFrame.head](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html?highlight=head#pandas.DataFrame.head)

In [ ]:
# 学習データの先頭5行を表示
train.head()

In [ ]:
# テストデータの先頭5行を表示
test.head()

In [ ]:
# 提出サンプルデータの先頭5行を表示
sample.head()

学習用データを特徴量と目的変数に分けます。この際、特徴量には_x、目的変数には_yをつけました。目的変数は'Survived'です。テストデータには目的変数はなく特徴量だけなので分ける必要はないです。学習用データと同じようにテストデータには_xをつけました。<br>
[pd.DataFrame.drop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html?highlight=drop#pandas.DataFrame.drop)

In [ ]:
# 学習用データを特徴量と目的変数に分ける
train_x = train.drop(['Survived'], axis=1)
train_y = train['Survived']

# テストデータは特徴量のみなのでそのままで良い
test_x = test.copy()

'Survived'の分布を確認します。<br>
[sns.countplot](https://seaborn.pydata.org/generated/seaborn.countplot.html?highlight=countplot#seaborn.countplot)

In [ ]:
# 'Survived'の分布
sns.countplot(data=train, x='Survived')

# 2. 特徴量の作成

'PassengerId'は乗客に番号を振っているだけであり、目的変数に影響を与えないため、削除します。

In [ ]:
train_x = train_x.drop(['PassengerId'], axis=1)
test_x = test_x.drop(['PassengerId'], axis=1)

'Name', 'Ticket', 'Cabin'も上手く使えば予測に有用ですが、煩雑な処理 が必要そうなので、今回はこれらの変数を使わないことにします。

In [ ]:
drop_col = ['Name','Ticket', 'Cabin']
train_x = train_x.drop(drop_col, axis=1)
test_x = test_x.drop(drop_col, axis=1)

学習用データとテストデータの欠損値を確認します。train_x.isnull().sum()とすると、学習用データの特徴量ごとの欠損数を出力することができます。さらにtrain_x.isnull().sum().sort_values(ascending=False)を加えることで、欠損数が多い順に並び替えることができます。<br>
[pd.DataFrame.isnull](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isnull.html?highlight=isnull#pandas.DataFrame.isnull)
[pd.Series.sort_values](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_values.html?highlight=sort_value)

In [ ]:
#学習データの欠損値を確認する
print('訓練データの欠損値:\n', train_x.isnull().sum().sort_values(ascending=False), '\n')
#テストデータの欠損値を確認する
print('テストデータの欠損値:\n', test_x.isnull().sum().sort_values(ascending=False))

'Age', 'Embarked', 'Fare'は欠損値があるため補完します。まず'Age', 'Embarked', 'Fare'のデータ型を確認します。

In [ ]:
# データの情報の確認
print(train_x.info())

'Age'は数値変数、'Embarked'はカテゴリ変数、'Fare'は数値変数です。数値変数であるAge, Fareは平均値、カテゴリ変数であるEmbarkedは最頻値で補完します。<br>
[pd.Series.fillna](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html?highlight=fillna#pandas.Series.fillna)
[pd.Series.mean](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html?highlight=series%20mean#pandas.Series.mean)

In [ ]:
# Ageカラムの欠損値を平均値で補完する
train_x['Age'] = train_x['Age'].fillna(train_x['Age'].mean())
test_x['Age'] = test_x['Age'].fillna(test_x['Age'].mean())

In [ ]:
# 'Embarked'の分布
sns.countplot(data=train_x, x='Embarked')

In [ ]:
# Embarkedカラムの欠損値を最頻値で補完する
train_x['Embarked'] = train_x['Embarked'].fillna('S')
test_x['Embarked'] = test_x['Embarked'].fillna('S')

In [ ]:
# Fareカラムの欠損値を平均値で補完する
train_x['Fare'] = train_x['Fare'].fillna(train_x['Fare'].mean())
test_x['Fare'] = test_x['Fare'].fillna(test_x['Fare'].mean())

In [ ]:
# 欠損値の処理が完了したことを確認する
print(train_x.isnull().sum(), '\n')
print(test_x.isnull().sum())

以上で欠損値の処理が完了しました。次はカテゴリ変数の変換をします。今回はラベルエンコーディングします。まずデータの情報を確認します。

In [ ]:
# データの情報の確認
print(train_x.info(), '\n')
print(test_x.info())

Dtypeがobjectとなっている変数がカテゴリ変数です。したがって、'Sex'についてラベルエンコーディングする必要があります。'Sex'に含まれるカテゴリを確認します。<br>
[pd.Series.unique](https://pandas.pydata.org/docs/reference/api/pandas.Series.unique.html?highlight=series%20unique#pandas.Series.unique)

In [ ]:
print(train_x['Sex'].unique())
print(test_x['Sex'].unique())

'Sex'カラムにはmale, femaleが含まれることが分かりました。maleは0、femaleは1に変換したいと思います。<br>
[pd.Series.map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html?highlight=map#pandas.Series.map)

In [ ]:
# 'Sex'をマッピング　male:0, female:1
sex_mapping = {"male":0, "female":1}
train_x["Sex"] = train_x["Sex"].map(sex_mapping)
test_x["Sex"] = test_x["Sex"].map(sex_mapping)

次に'Embarked'カラムについてラベルエンコーディングします。'Embarked'に含まれるカテゴリを確認します。

In [ ]:
print(train_x['Embarked'].unique())
print(test_x['Embarked'].unique())

'Embarked'カラムにはS, C, Qが含まれることがわかりました。Sは0、Cは1、Qは2に変換したいと思います。

In [ ]:
# Embarkedをマッピング　S:0, C:1, Q:2
embarked_mapping = {'S':0, 'C':1, 'Q':2}
train_x['Embarked'] = train_x['Embarked'].map(embarked_mapping)
test_x['Embarked'] = test_x['Embarked'].map(embarked_mapping)

再びデータの情報を確認します。

In [ ]:
# データの情報の確認
print(train_x.info(), '\n')
print(test_x.info())

全ての変数を数値変数に変換できたことが分かります。

# 3. モデリング

性能を評価するには、データの分割をする必要があります。今回はホールドアウト検証を使います。<br>
[sklearn.model_selection.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html#sklearn.model_selection.train_test_split)

In [ ]:
# 分割方法の指定
tr_x, va_x, tr_y, va_y = train_test_split(train_x, train_y, test_size=0.3, random_state=0)

例としてRandomForestClassifierを用いてモデルの作成を行いたいと思います。<br>
[sklearn.ensemble.RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html?highlight=randomforestclassifier#sklearn.ensemble.RandomForestClassifier)<br>

性能指標にはaccuracyを用います。<br>
[sklearn.metrics.accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html#sklearn.metrics.accuracy_score)

In [ ]:
# モデルの作成と評価(RandomForestClassifier)
rfc = RandomForestClassifier(random_state=0)
rfc.fit(tr_x, tr_y)
va_pred = rfc.predict(va_x)
score = accuracy_score(va_y, va_pred)
score

# 4. 提出物の作成

In [ ]:
# 学習データ全体でモデルの学習をする
rfc.fit(train_x, train_y)

# テストデータに対して予測する
predict = rfc.predict(test_x)
predict

[pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html?highlight=dataframe#pandas.DataFrame)

In [ ]:
# 提出用の変数の作成
submit = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predict})
submit

[pd.DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html?highlight=to_csv#pandas.DataFrame.to_csv)

In [ ]:
# 提出用ファイルの作成
submit.to_csv('submission.csv', index=False)